# VisioStock · Entrenamiento del detector de cajas (YOLOv8n)

Este cuaderno entrena la "cámara" de VisioStock para detectar **cajas** y genera `cajas.pt`, el modelo que usa `vision/` (HU-05/HU-06).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

| Paso | Qué hace |
| --- | --- |
| 1 | Revisa la GPU e instala Ultralytics |
| 2 | Conecta Google Drive (ahí se guarda todo, por si Colab se desconecta) |
| 3 | Carga el dataset: **A) Roboflow** o **B) un .zip propio** en formato YOLO |
| 4 | Deja el dataset con una sola clase: `caja` |
| 5 | Entrena |
| 6 | Revisa las métricas (meta: precisión de conteo ≥ 85 %) |
| 7 | Prueba en un video con seguimiento y conteo por línea |
| 8 | Guarda `cajas.pt` en Drive y lo descarga |

## 1 · GPU e instalación

In [ ]:
!nvidia-smi
%pip install -q ultralytics roboflow

In [ ]:
import ultralytics
ultralytics.checks()

## 2 · Google Drive y configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

CARPETA_DRIVE = Path('/content/drive/MyDrive/visiostock')  # aquí quedan los entrenamientos y el modelo final
CARPETA_DATOS = Path('/content/datos')                      # dataset (en el disco local de Colab, es más rápido)
CARPETA_DRIVE.mkdir(parents=True, exist_ok=True)

# Parámetros del entrenamiento
MODELO_BASE = 'yolov8n.pt'  # n = el más liviano; probar 'yolov8s.pt' si la precisión no alcanza
EPOCAS = 100
TAMANO_IMG = 640
LOTE = 16                    # bajar a 8 si sale "CUDA out of memory"
PACIENCIA = 20               # corta antes si no mejora en 20 épocas
NOMBRE_RUN = 'cajas'

FUENTE_DATOS = 'roboflow'    # 'roboflow' (opción A) o 'zip' (opción B)

## 3 · Dataset

### Opción A · Roboflow
1. En [Roboflow Universe](https://universe.roboflow.com) busca un dataset de cajas (p. ej. *"cardboard box"*, *"carton box detection"*) o sube y etiqueta las fotos del almacén en tu propio proyecto.
2. `Download Dataset → YOLOv8 → show download code` y copia `workspace`, `project` y `version` abajo.
3. Tu API key está en *Settings → API Keys*. Se pide con un cuadro oculto: no la pegues en el código.

In [ ]:
if FUENTE_DATOS == 'roboflow':
    from getpass import getpass
    from roboflow import Roboflow

    RF_WORKSPACE = 'tu-workspace'   # <- cambiar
    RF_PROYECTO = 'tu-proyecto'     # <- cambiar
    RF_VERSION = 1                  # <- cambiar

    rf = Roboflow(api_key=getpass('API key de Roboflow: '))
    dataset = rf.workspace(RF_WORKSPACE).project(RF_PROYECTO).version(RF_VERSION).download(
        'yolov8', location=str(CARPETA_DATOS), overwrite=True
    )
    print('Descargado en', CARPETA_DATOS)

### Opción B · .zip propio
Un .zip en formato YOLO (exportado de Roboflow, CVAT o Label Studio). Debe tener `images/` y `labels/`, con o sin carpetas `train/valid/test`:
```
datos.zip
├── train/images/*.jpg   train/labels/*.txt
└── valid/images/*.jpg   valid/labels/*.txt      (si no hay valid, se crea con el 20 % de train)
```
Puedes subirlo desde tu PC o dejarlo en Drive y poner la ruta en `ZIP_EN_DRIVE`.

In [ ]:
if FUENTE_DATOS == 'zip':
    import shutil, zipfile

    ZIP_EN_DRIVE = ''  # p. ej. '/content/drive/MyDrive/visiostock/datos.zip'; vacío = subir desde la PC

    if ZIP_EN_DRIVE:
        ruta_zip = Path(ZIP_EN_DRIVE)
    else:
        from google.colab import files
        subido = files.upload()
        ruta_zip = Path(next(iter(subido)))

    shutil.rmtree(CARPETA_DATOS, ignore_errors=True)
    with zipfile.ZipFile(ruta_zip) as z:
        z.extractall(CARPETA_DATOS)
    print('Extraído en', CARPETA_DATOS)

## 4 · Preparar el dataset con una sola clase `caja`
Busca las carpetas de imágenes, pasa todas las etiquetas a la clase 0 (`caja`), crea la validación si falta y escribe `data.yaml`.

In [ ]:
import random, shutil, yaml

EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def carpetas_de_imagenes(raiz: Path):
    """Devuelve {split: carpeta_images} buscando .../<split>/images o .../images."""
    encontradas = {}
    for img_dir in sorted(raiz.rglob('images')):
        if not img_dir.is_dir():
            continue
        padre = img_dir.parent.name.lower()
        split = {'valid': 'val', 'validation': 'val'}.get(padre, padre)
        if split not in ('train', 'val', 'test'):
            split = 'train'
        encontradas.setdefault(split, img_dir)
    return encontradas

def etiquetas_de(img_dir: Path) -> Path:
    return img_dir.parent / 'labels'

def a_una_clase(lbl_dir: Path) -> int:
    """Cambia la clase de cada caja a 0. Devuelve cuántas cajas hay."""
    total = 0
    for txt in lbl_dir.glob('*.txt'):
        lineas = []
        for linea in txt.read_text().splitlines():
            partes = linea.split()
            if len(partes) >= 5:
                lineas.append(' '.join(['0'] + partes[1:]))
        txt.write_text('\n'.join(lineas) + ('\n' if lineas else ''))
        total += len(lineas)
    return total

def crear_validacion(train_img: Path, fraccion=0.2, semilla=42) -> Path:
    val_img = CARPETA_DATOS / 'val' / 'images'
    val_lbl = CARPETA_DATOS / 'val' / 'labels'
    val_img.mkdir(parents=True, exist_ok=True)
    val_lbl.mkdir(parents=True, exist_ok=True)
    imagenes = sorted(p for p in train_img.iterdir() if p.suffix.lower() in EXT_IMG)
    random.Random(semilla).shuffle(imagenes)
    for img in imagenes[: max(1, int(len(imagenes) * fraccion))]:
        shutil.move(str(img), val_img / img.name)
        lbl = etiquetas_de(train_img) / (img.stem + '.txt')
        if lbl.exists():
            shutil.move(str(lbl), val_lbl / lbl.name)
    return val_img

splits = carpetas_de_imagenes(CARPETA_DATOS)
assert 'train' in splits, f'No se encontró una carpeta images/ en {CARPETA_DATOS}'
if 'val' not in splits:
    print('No hay validación: se separa el 20 % de train.')
    splits['val'] = crear_validacion(splits['train'])

for split, img_dir in splits.items():
    n_img = sum(1 for p in img_dir.iterdir() if p.suffix.lower() in EXT_IMG)
    n_cajas = a_una_clase(etiquetas_de(img_dir))
    print(f'{split:>5}: {n_img} imágenes, {n_cajas} cajas etiquetadas')

DATA_YAML = CARPETA_DATOS / 'data_visiostock.yaml'
DATA_YAML.write_text(yaml.safe_dump({
    'path': str(CARPETA_DATOS),
    **{k: str(v) for k, v in splits.items()},
    'nc': 1,
    'names': ['caja'],
}, allow_unicode=True))
print(DATA_YAML.read_text())

Revisión visual de algunas etiquetas (las cajas deben quedar bien encuadradas):

In [ ]:
import cv2
import matplotlib.pyplot as plt

muestras = random.sample(sorted(p for p in splits['train'].iterdir() if p.suffix.lower() in EXT_IMG), k=6)
fig, ejes = plt.subplots(2, 3, figsize=(15, 9))
for eje, img_path in zip(ejes.flat, muestras):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = etiquetas_de(splits['train']) / (img_path.stem + '.txt')
    for linea in (lbl.read_text().splitlines() if lbl.exists() else []):
        _, cx, cy, bw, bh = map(float, linea.split()[:5])
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), max(2, w // 300))
    eje.imshow(img); eje.set_title(img_path.name, fontsize=8); eje.axis('off')
plt.tight_layout(); plt.show()

## 5 · Entrenamiento
Los resultados se guardan en Drive (`visiostock/runs/cajas`). Si Colab se desconecta, corre la celda **Reanudar**.

In [ ]:
from ultralytics import YOLO

modelo = YOLO(MODELO_BASE)
resultado = modelo.train(
    data=str(DATA_YAML),
    epochs=EPOCAS,
    imgsz=TAMANO_IMG,
    batch=LOTE,
    patience=PACIENCIA,
    project=str(CARPETA_DRIVE / 'runs'),
    name=NOMBRE_RUN,
    exist_ok=True,
    seed=42,
    # aumentos útiles en almacén: cambios de luz, cajas giradas o parcialmente tapadas
    hsv_v=0.5,
    degrees=5,
    erasing=0.4,
)
CARPETA_RUN = Path(resultado.save_dir)
print('Entrenamiento guardado en', CARPETA_RUN)

In [ ]:
# Reanudar (solo si el entrenamiento se cortó)
# CARPETA_RUN = CARPETA_DRIVE / 'runs' / NOMBRE_RUN
# YOLO(str(CARPETA_RUN / 'weights' / 'last.pt')).train(resume=True)

## 6 · Métricas

In [ ]:
from IPython.display import Image, display

CARPETA_RUN = CARPETA_DRIVE / 'runs' / NOMBRE_RUN
MEJOR = CARPETA_RUN / 'weights' / 'best.pt'

for grafico in ('results.png', 'confusion_matrix.png', 'PR_curve.png', 'val_batch0_pred.jpg'):
    ruta = CARPETA_RUN / grafico
    if ruta.exists():
        print(grafico); display(Image(filename=str(ruta), width=900))

In [ ]:
metricas = YOLO(str(MEJOR)).val(data=str(DATA_YAML), imgsz=TAMANO_IMG, split='val')
p, r = metricas.box.mp, metricas.box.mr
print(f'Precisión: {p:.1%}   Recall: {r:.1%}   mAP50: {metricas.box.map50:.1%}   mAP50-95: {metricas.box.map:.1%}')
print('Recall ≥ 85 %: el detector ve casi todas las cajas.' if r >= 0.85
      else 'Recall < 85 %: faltan cajas por detectar. Agregar fotos del almacén real, más épocas o probar yolov8s.pt.')

## 7 · Prueba en video con seguimiento y conteo
Sube un video corto del almacén. Se usa la misma lógica que `vision/contador.py`: una caja que cruza la línea de arriba hacia abajo es **entrada**; de abajo hacia arriba, **salida**. Si sabes cuántas cajas pasaron realmente, ponlas en `REAL_ENTRADAS` / `REAL_SALIDAS` para calcular la **precisión de conteo** (la misma fórmula de `vision/evaluar.py`).

In [ ]:
from google.colab import files

LINEA_Y = 0.5          # fracción del alto del cuadro
CONFIANZA_MIN = 0.4
REAL_ENTRADAS = None   # p. ej. 12 (conteo manual del video)
REAL_SALIDAS = None    # p. ej. 4

subido = files.upload()
VIDEO = Path(next(iter(subido)))

In [ ]:
detector = YOLO(str(MEJOR))
captura = cv2.VideoCapture(str(VIDEO))
fps = captura.get(cv2.CAP_PROP_FPS) or 30
ancho, alto = int(captura.get(cv2.CAP_PROP_FRAME_WIDTH)), int(captura.get(cv2.CAP_PROP_FRAME_HEIGHT))
SALIDA = Path('/content/conteo_' + VIDEO.stem + '.mp4')
escritor = cv2.VideoWriter(str(SALIDA), cv2.VideoWriter_fourcc(*'mp4v'), fps, (ancho, alto))

linea = LINEA_Y * alto
ultima_y, contados = {}, set()
entradas = salidas = 0

while True:
    ok, cuadro = captura.read()
    if not ok:
        break
    res = detector.track(cuadro, conf=CONFIANZA_MIN, tracker='bytetrack.yaml', persist=True, verbose=False)[0]
    if res.boxes is not None and res.boxes.id is not None:
        for (x1, y1, x2, y2), tid in zip(res.boxes.xyxy.tolist(), res.boxes.id.int().tolist()):
            y = (y1 + y2) / 2
            anterior = ultima_y.get(tid)
            ultima_y[tid] = y
            if anterior is not None and tid not in contados:
                if anterior < linea <= y:
                    entradas += 1; contados.add(tid)
                elif anterior >= linea > y:
                    salidas += 1; contados.add(tid)
            cv2.rectangle(cuadro, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
            cv2.putText(cuadro, f'#{tid}', (int(x1), int(y1) - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.line(cuadro, (0, int(linea)), (ancho, int(linea)), (0, 200, 255), 2)
    cv2.putText(cuadro, f'Entradas: {entradas}  Salidas: {salidas}', (10, alto - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    escritor.write(cuadro)

captura.release(); escritor.release()
print(f'Entradas: {entradas}   Salidas: {salidas}')

if REAL_ENTRADAS is not None and REAL_SALIDAS is not None:
    total = REAL_ENTRADAS + REAL_SALIDAS
    error = abs(entradas - REAL_ENTRADAS) + abs(salidas - REAL_SALIDAS)
    precision = max(0.0, 1 - error / total) if total else float(entradas + salidas == 0)
    print(f'Precisión de conteo: {precision:.1%} ->', 'cumple' if precision >= 0.85 else 'no cumple', 'la meta de 85 %')

In [ ]:
# Ver el video con el conteo (se convierte a H.264 para que el navegador lo reproduzca)
from base64 import b64encode
from IPython.display import HTML

WEB = SALIDA.with_name(SALIDA.stem + '_web.mp4')
!ffmpeg -y -loglevel error -i "{SALIDA}" -vcodec libx264 -pix_fmt yuv420p "{WEB}"
HTML(f'<video width=800 controls src="data:video/mp4;base64,{b64encode(WEB.read_bytes()).decode()}"></video>')

## 8 · Guardar el modelo
Copia `best.pt` como `cajas.pt` en Drive y lo descarga. Luego, en el repo:
1. Pon `cajas.pt` dentro de `vision/` (está en `.gitignore`: **no se sube a GitHub**, compártelo por Drive).
2. En `vision/.env`: `MODELO=cajas.pt` y `CLASES=caja`.
3. Prueba: `python main.py --fuente video.mp4 --mostrar --sin-envio`.

In [ ]:
import shutil
from google.colab import files

FINAL = CARPETA_DRIVE / 'cajas.pt'
shutil.copy(MEJOR, FINAL)
print('Guardado en Drive:', FINAL)
files.download(str(FINAL))